# PyTorch 第四章：模型模块

> 核心目标：理解一个 PyTorch 模型如何**定义、组织、管理参数、执行 forward、提取中间特征，并初始化权重**。

本 Notebook 按《PyTorch 实用教程（第二版）》第四章顺序整理，保留核心内容并做精简：

1. **4.1 Module & Parameter**
2. **4.2 Module 容器**
3. **4.3 常用网络层**
4. **4.4 Module 常用 API**
5. **4.5 Hook 与 Grad-CAM**
6. **4.6 经典模型代码分析**
7. **4.7 权重初始化**

参考：
- https://tingsongyu.github.io/PyTorch-Tutorial-2nd/chapter-4/
- https://docs.pytorch.org/docs/stable/nn.html
- https://pytorch.org/vision/stable/models.html


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import models

torch.manual_seed(42)

print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())


## 4.1 Module & Parameter

PyTorch 中，模型、网络层、子模块基本都继承自 `nn.Module`。

自定义模型通常只需要三步：

```text
1. 继承 nn.Module
2. __init__ 中创建网络层
3. forward 中定义数据怎么流动
```

Module 会自动管理：

- `_modules`：子 Module
- `_parameters`：可训练 Parameter
- `_buffers`：不参与梯度更新、但属于模型状态的数据
- hooks：前向/反向钩子

`Parameter` 本质上是一种特殊 Tensor，用来告诉 Module：

> **这是模型需要管理和训练的参数。**

注意：

- **参数 parameter**：通过训练更新，如 weight、bias
- **超参数 hyperparameter**：人为设置，如 kernel size、learning rate


In [ ]:
class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()

        # 赋值给 self.xxx 后，Module 会自动注册这些子模块
        self.fc1 = nn.Linear(4, 8)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(8, 2)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        return self.fc2(x)

model = TinyNet()

x = torch.randn(3, 4)
y = model(x)

print("output shape:", y.shape)
print("_modules keys:", list(model._modules.keys()))


In [ ]:
# Parameter 与普通 Tensor 的区别
class ScaleLayer(nn.Module):
    def __init__(self):
        super().__init__()

        self.scale = nn.Parameter(torch.tensor(2.0))  # 会被 optimizer 管理
        self.offset = torch.tensor(1.0)               # 普通 Tensor，不是 Parameter

    def forward(self, x):
        return x * self.scale + self.offset

layer = ScaleLayer()

print("parameters:")
for name, p in layer.named_parameters():
    print(name, p)

print("\nscale 是否在 parameters 中:", "scale" in dict(layer.named_parameters()))
print("offset 是否在 parameters 中:", "offset" in dict(layer.named_parameters()))


### Buffer：属于模型，但不训练

有些状态需要跟着模型一起：

- 保存到 `state_dict`
- 随模型移动到 CPU / GPU

但它们不需要梯度更新，这时使用 `register_buffer()`。

典型例子：BatchNorm 的 `running_mean`、`running_var`。


In [ ]:
class DemoBuffer(nn.Module):
    def __init__(self):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(1))
        self.register_buffer("running_value", torch.zeros(1))

m = DemoBuffer()

print("Parameter:", list(m.named_parameters()))
print("Buffer   :", list(m.named_buffers()))
print("state_dict keys:", list(m.state_dict().keys()))


## 4.2 Module 容器

容器用于把多个 Module 组织起来。

最重要的三个：

| 容器 | 特点 |
|---|---|
| `nn.Sequential` | 按固定顺序自动执行 |
| `nn.ModuleList` | 像 list 管理 Module，但 forward 由自己写 |
| `nn.ModuleDict` | 像 dict 管理 Module，可按名字选择 |

入门阶段优先掌握：

> **固定流水线 → Sequential；动态/循环结构 → ModuleList。**


In [ ]:
# Sequential：输入会自动按顺序经过每一层
seq = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 2)
)

x = torch.randn(3, 4)
print(seq(x).shape)
print(seq)


In [ ]:
# ModuleList：只负责“注册和管理”，不会自动定义 forward
class DeepMLP(nn.Module):
    def __init__(self, depth=3):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(4, 4) for _ in range(depth)
        ])

    def forward(self, x):
        for layer in self.layers:
            x = F.relu(layer(x))
        return x

model = DeepMLP(depth=3)

print(model(torch.randn(2, 4)).shape)
print("注册的参数量:", sum(p.numel() for p in model.parameters()))


### 为什么不要用普通 Python list 存网络层？

```python
self.layers = [nn.Linear(...), ...]
```

普通 list 中的层不会像 `ModuleList` 那样被 Module 正确注册，因此可能：

- 不出现在 `model.parameters()`
- 不跟随 `model.to(device)`
- 不进入 `state_dict()`

同理还有 `ModuleDict / ParameterList / ParameterDict`。


## 4.3 常用网络层

教程列出的网络层很多，学习时不要背 API。

建议掌握这些高频类别：

```text
Conv2d          → 提取空间特征
Pooling         → 降低空间尺寸
AdaptivePool    → 固定输出尺寸
Linear          → 全连接映射
BatchNorm       → 标准化
Dropout         → 随机失活
Activation      → 引入非线性
```

学习一个网络层时始终问四个问题：

1. 输入 shape 是什么？
2. 输出 shape 是什么？
3. 有哪些可训练参数？
4. `train()` 和 `eval()` 时行为是否不同？


In [ ]:
# Conv2d：重点观察 shape
conv = nn.Conv2d(
    in_channels=3,
    out_channels=16,
    kernel_size=3,
    stride=2,
    padding=1
)

x = torch.randn(8, 3, 32, 32)
y = conv(x)

print("input :", x.shape)
print("output:", y.shape)
print("weight:", conv.weight.shape)  # [out_channels, in_channels, kH, kW]


### 卷积输出尺寸

普通二维卷积单个空间维度：

$$
H_{out}
=
\left\lfloor
\frac{H_{in}+2p-d(k-1)-1}{s}+1
\right\rfloor
$$

其中：

- $k$：kernel size
- $s$：stride
- $p$：padding
- $d$：dilation

实际写网络时，除了会公式，更重要的是**随时打印 shape 验证**。


In [ ]:
# Pooling 与 Adaptive Pooling
x = torch.randn(2, 16, 32, 32)

pool = nn.MaxPool2d(kernel_size=2, stride=2)
adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))

x1 = pool(x)
x2 = adaptive_pool(x1)

print("original:", x.shape)
print("MaxPool :", x1.shape)
print("Adaptive:", x2.shape)


In [ ]:
# 一个典型 CNN block：Conv + BN + ReLU
block = nn.Sequential(
    nn.Conv2d(3, 16, kernel_size=3, padding=1),
    nn.BatchNorm2d(16),
    nn.ReLU()
)

x = torch.randn(4, 3, 32, 32)
print(block(x).shape)

# BN 中 weight / bias 是 Parameter
bn = block[1]
print("BN weight shape:", bn.weight.shape)
print("running_mean 是 Parameter 吗:",
      "running_mean" in dict(bn.named_parameters()))
print("running_mean 是 Buffer 吗:",
      "running_mean" in dict(bn.named_buffers()))


### BatchNorm 与 Dropout：必须理解 train / eval

这两类层在训练和推理阶段行为不同。

**BatchNorm**

- `train()`：使用当前 batch 统计量，并更新 running statistics
- `eval()`：使用保存的 running statistics

**Dropout**

- `train()`：随机失活
- `eval()`：不再随机失活

因此验证/推理前通常要：

```python
model.eval()
```

但注意：`model.eval()` **不会关闭梯度记录**。
纯推理通常还配合：

```python
with torch.inference_mode():
    ...
```


In [ ]:
dropout = nn.Dropout(p=0.5)
x = torch.ones(10)

dropout.train()
print("train:", dropout(x))

dropout.eval()
print("eval :", dropout(x))


## 4.4 Module 常用 API

最值得掌握的几组：

### 模式
- `model.train()`
- `model.eval()`

### 设备 / 精度
- `model.to(device)`
- `model.float() / half() / bfloat16()`

### 参数与模块
- `parameters() / named_parameters()`
- `modules() / named_modules()`
- `children() / named_children()`

### 权重
- `state_dict()`
- `load_state_dict()`

### 批量操作
- `apply(fn)`


In [ ]:
model = TinyNet()

print("=== named_parameters ===")
for name, p in model.named_parameters():
    print(name, tuple(p.shape))

print("\n=== named_children ===")
for name, module in model.named_children():
    print(name, "->", module.__class__.__name__)

print("\n=== state_dict ===")
for key, value in model.state_dict().items():
    print(key, tuple(value.shape))


In [ ]:
# state_dict：保存的是模型状态，不是 forward 逻辑本身
model_a = TinyNet()
model_b = TinyNet()

# 把 A 的参数复制到 B
result = model_b.load_state_dict(model_a.state_dict())

print(result)

# 检查对应参数是否相同
same = all(
    torch.equal(a, b)
    for a, b in zip(model_a.parameters(), model_b.parameters())
)
print("参数完全一致:", same)


`load_state_dict()` 常见报错：

```text
Missing key(s)
Unexpected key(s)
size mismatch
```

本质都是：

> **当前模型结构和待加载的 state_dict 对不上。**

`strict=False` 可以允许部分 key 不匹配，但不能盲目使用；要先确认哪些权重没有成功加载。


## 4.5 Hook：不修改 forward 也能观察中间结果

Hook 的核心思想：

> **给模型某个位置挂一个回调函数，在 forward / backward 发生时自动执行。**

教程重点介绍：

- `Tensor.register_hook`
- `Module.register_forward_pre_hook`
- `Module.register_forward_hook`
- `Module.register_full_backward_hook`

最常见用途：

- 获取中间层特征图
- 获取梯度
- 调试网络
- 实现 Grad-CAM 等可视化方法


In [ ]:
# forward hook：提取中间层特征图
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1, 4, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(4, 2)

    def forward(self, x):
        x = self.relu(self.conv(x))
        x = self.pool(x).flatten(1)
        return self.fc(x)

model = SmallCNN()
feature_maps = []

def save_feature(module, inputs, output):
    feature_maps.append(output.detach())

handle = model.conv.register_forward_hook(save_feature)

x = torch.randn(2, 1, 8, 8)
out = model(x)

print("model output:", out.shape)
print("conv feature:", feature_maps[0].shape)

# hook 用完应该移除
handle.remove()


In [ ]:
# Tensor hook：捕获非叶子 Tensor 的梯度
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x * 2

saved_grad = []

handle = y.register_hook(lambda grad: saved_grad.append(grad.clone()))

loss = (y ** 2).mean()
loss.backward()

print("y.grad:", y.grad)               # 默认通常不保存
print("hook 捕获的梯度:", saved_grad[0])

handle.remove()


### Grad-CAM 为什么需要 Hook？

Grad-CAM 的核心需要某个卷积层的：

1. **feature map（前向激活）**
2. **该 feature map 对目标类别的梯度**

因此可以：

```text
forward hook  → 保存 feature map
backward hook → 保存 gradient
               ↓
        加权组合得到热力图
```

当前阶段先掌握 Hook 的机制即可；完整 Grad-CAM 可视化会在后续可视化章节再次出现。


## 4.6 经典模型代码分析

教程分析了 AlexNet、VGG、GoogLeNet、ResNet。

这一节不要背网络结构，真正要学习的是**复杂模型怎样被组织成代码**：

| 模型 | 最值得学习的代码思想 |
|---|---|
| AlexNet | `features → flatten → classifier` |
| VGG | 用配置表 + `make_layers()` 批量构建网络 |
| GoogLeNet | 把重复的 Inception 结构抽象成独立 Module |
| ResNet | 把 BasicBlock / Bottleneck 抽象成可重复模块 |

复杂网络的共同原则：

> **重复结构抽象成 Module，再通过容器和配置进行组合。**


In [ ]:
# 使用当前 torchvision API，weights=None 表示不下载预训练权重
alexnet = models.alexnet(weights=None)
vgg11 = models.vgg11(weights=None)
resnet18 = models.resnet18(weights=None)

print("AlexNet 的大模块:")
for name, module in alexnet.named_children():
    print(" ", name, "->", module.__class__.__name__)

print("\nVGG11 的大模块:")
for name, module in vgg11.named_children():
    print(" ", name, "->", module.__class__.__name__)

print("\nResNet18 的大模块:")
for name, module in resnet18.named_children():
    print(" ", name, "->", module.__class__.__name__)


In [ ]:
# ResNet 最核心的思想：残差连接
class SimpleResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        identity = x
        out = self.block(x)

        # 核心：F(x) + x
        out = out + identity
        return self.relu(out)

block = SimpleResidualBlock(8)
x = torch.randn(2, 8, 16, 16)

print(block(x).shape)


读 torchvision 模型源码时，推荐顺序：

```text
1. 先看 forward
2. 再看 __init__ 中 forward 用到的属性
3. 遇到重复模块，再进入对应 Block
4. 最后看构建函数 / 配置表
```

不要一开始从文件第一行逐行读到最后。

例如 ResNet：

```text
resnet18()
   ↓
ResNet
   ↓
_make_layer()
   ↓
BasicBlock × N
```

这样读会比逐行阅读高效很多。


## 4.7 权重初始化

教程重点介绍：

- Xavier / Glorot 初始化
- Kaiming / He 初始化
- normal / uniform / constant
- zeros / ones / orthogonal 等

入门阶段重点理解：

### Xavier
常用于 sigmoid / tanh 等场景的经典初始化思想。

### Kaiming
针对 ReLU 系列激活函数设计，CNN 中非常常见。

实际项目中通常：

> **按网络层类型遍历模型，然后选择对应初始化方法。**


In [ ]:
class InitDemo(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 4)
        )

    def forward(self, x):
        return self.net(x)

model = InitDemo()

def init_weights(module):
    if isinstance(module, nn.Linear):
        # ReLU 网络常用 Kaiming 初始化
        nn.init.kaiming_normal_(
            module.weight,
            mode="fan_in",
            nonlinearity="relu"
        )

        if module.bias is not None:
            nn.init.zeros_(module.bias)

# apply 会递归地对所有子模块执行函数
model.apply(init_weights)

for name, p in model.named_parameters():
    print(name, "mean =", round(p.detach().mean().item(), 4))


### 一个常见误区：不要把所有权重初始化成 0

如果同一层所有神经元拥有完全相同的初始化和计算过程，它们会产生对称行为，很难学习出不同特征。

因此：

- `weight` 通常使用随机初始化（Kaiming / Xavier 等）
- `bias` 常可以初始化为 0

而且 PyTorch 各网络层本身已经提供合理的默认初始化。

> 不要为了“用了初始化方法”而无条件覆盖默认初始化。


# 第四章总结

本章真正需要掌握的是这一条逻辑：

```text
nn.Module
   │
   ├── 子 Module → _modules
   ├── Parameter → _parameters
   ├── Buffer    → _buffers
   │
   ↓
forward()
   ↓
模型计算
   │
   ├── Hook → 观察中间特征 / 梯度
   └── state_dict → 保存模型状态
```

必须会回答：

1. 为什么自定义模型要继承 `nn.Module`？
2. `__init__()` 和 `forward()` 分别负责什么？
3. `Parameter` 和普通 Tensor 有什么区别？
4. Buffer 与 Parameter 有什么区别？
5. `Sequential` 与 `ModuleList` 有什么区别？
6. Conv2d 的 `in_channels / out_channels / kernel_size / stride / padding` 分别是什么？
7. 为什么 BatchNorm 和 Dropout 必须区分 `train()` / `eval()`？
8. `state_dict()` 保存的是什么？
9. Hook 为什么能够提取中间层特征？
10. VGG / ResNet 的源码为什么要抽象重复 Module？
11. Kaiming 和 Xavier 初始化解决什么问题？

### 最核心的建模模板

```python
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        # 创建并注册子模块

    def forward(self, x):
        # 定义数据流
        return x
```

如果你已经能自己搭一个 `Conv → BN → ReLU → Pool → Linear` 网络，并能解释参数是如何被 Module 管理的，就可以进入第五章优化模块。
